#Imagen por resonancia magnética funcional (fMRI)
La imagen por resonancia magnética funcional (fMRI) es una técnica de neuroimagen que mide los cambios en el flujo sanguíneo para mostrar qué áreas del cerebro están activas en tiempo real

In [ ]:
import requests
import zipfile
import pandas as pd
import numpy as np
import os
import nibabel
import matplotlib.pyplot as plt

# Enable plots inside the Jupyter Notebook
%matplotlib inline

El procedimiento se realiza en el mismo resonador utilizado para obtener imágenes anatómicas por resonancia magnética para diagnóstico, pero con modificaciones especiales del software y del hardware. Para realizar una IRMf no se requiere necesariamente inyecciones de sustancia alguna ni radiación ionizante.
(Wikipedia)

 El aumento de actividad en las áreas cerebrales relacionadas con x tarea causa una vasodilatación y un aumento del flujo sanguíneo en estas mismas áreas. Este aumento del flujo/volumen es detectado por el resonador y normalmente representado en una imagen.

In [ ]:
# Define the URL of the data and download it using the Requests libary
url = 'http://www.fil.ion.ucl.ac.uk/spm/download/data/MoAEpilot/MoAEpilot.zip'
data = requests.get(url)

Esta URL apunta al dataset MoAEpilot ("Mother of All Experiments" - pilot), un dataset clásico y muy usado para enseñanza, publicado por el FIL (Functional Imaging Laboratory) del UCL (University College London), los mismos creadores de SPM (Statistical Parametric Mapping, uno de los softwares más usados históricamente para analizar fMRI).

Consiste en un experimento auditivo simple: a un sujeto le hacen escuchar bloques de sonido alternados con silencio dentro del escáner, y se mide la respuesta BOLD (la señal que capta la fMRI) en la corteza auditiva.

In [ ]:
# Check if the targed folder for storing the data already exists. If not create it and save the zip file.
if os.path.exists('./fMRI_data') == False:
    os.mkdir('fMRI_data')

open('./fMRI_data/data.zip', 'wb').write(data.content)

# Un-zip the file
zip_ref = zipfile.ZipFile('./fMRI_data/data.zip', 'r')
zip_ref.extractall('./fMRI_data/')
zip_ref.close()

#Estudio de datos

In [ ]:
os.listdir('./fMRI_data/')

In [ ]:
open('./fMRI_data/README.txt').read()

In [ ]:
os.listdir('./fMRI_data/sM00223')

Lista todos los archivos dentro — pares de archivos .hdr + .img, el formato Analyze, que es el predecesor del NIfTI y que SPM usa mucho:
1. .hdr = header, contiene los metadatos (dimensiones, tipo de dato, orientación, matriz affine, etc.)
2. .img = contiene los datos crudos del array de intensidades (los vóxeles en sí)

Son dos archivos separados que juntos forman una sola imagen. (A diferencia de .nii, donde header + datos van en un solo archivo).

In [ ]:
os.listdir('./fMRI_data/fM00223')

In [ ]:
# Find all files in the structural data folder
data_path = './fMRI_data/sM00223/'
files = os.listdir(data_path)

# Read in the data
data_all = []
for data_file in files:
    if data_file[-3:] == 'hdr':    #filtra y carga solo lod hdr  , nibabel.load(...): carga el archivo y crea un objeto imagen de nibabe
        data = nibabel.load(data_path + data_file).get_fdata()   #get_fdata() trae los datos a memoria como un array de NumPy

In [ ]:
print(data.shape)

256x256 x 54 es la resolucion del volumen anatomico. 256x256 es la resolucion en el plano y 54 el numero de cortes en la 3 dimension (slices), formando asi un volumen 3D. La ultima dimension (1), determina el volumen temporal:


1. Foto (imagen estructural): aprietas el obturador una sola vez y obtienes una imagen fija. Así es el volumen sM00223 que cargamos, el escáner tomó un solo volumen 3D completo del cerebro, en un momento dado, y ya. No hay "antes" ni "después" en esa imagen, es estática.


2. Película (imagen funcional / fMRI): en vez de una sola foto, el escáner toma un volumen 3D completo cada pocos segundos, repetidamente, durante varios minutos, mientras el sujeto está dentro del escáner (por ejemplo, escuchando los bloques de sonido/silencio del experimento MoAEpilot). El resultado no es una imagen, sino una secuencia de imágenes 3D a lo largo del tiempo — como los "frames" de un video, pero cada "frame" es en sí mismo un volumen 3D completo del cerebro.


fMRI mide actividad cerebral indirectamente, a través de la señal BOLD (cambios en el flujo sanguíneo/oxigenación relacionados con actividad neuronal). Esa señal cambia con el tiempo — sube cuando una región se activa, baja cuando se desactiva. Para poder ver ese cambio, necesitas múltiples mediciones en el tiempo de la misma región, no una sola foto fija.

In [ ]:
print(files)

##Visualization

In [ ]:
data = data.squeeze()  # (256, 256, 54)
data

la variable data es un arreglo de numpy que se puede interpretar como una imagen 3D conformada por capas en cada uno de sus ejes. el eje x corresponde al eje sagital, el eje y al eje Coronal y el eje z al eje Axial, que corresponden a los planos anatomicos. La eleccion de la capa a visualizar se interpreta como un corte de la foto 3D en alguno de los planos.

In [ ]:
x = int(input("Slice 0: "))  #0-255
y = int(input("Slice 1: "))  #0-255
z = int(input("Slice 2: "))  #0-5

# Corte en el eje 0
slice_0 = data[x, :, :]   # forma (256, 54)

# Corte en el eje 1
slice_1 = data[:, y, :]   # forma (256, 54)

# Corte en el eje 2 (el más común para "ver la anatomía de arriba")
slice_2 = data[:, :, z]    # forma (256, 256)

fig, axes = plt.subplots(1, 3, figsize=(12, 5))

axes[0].imshow(data[x, :, :], cmap='gray', aspect=voxel_sizes[0]/voxel_sizes[2])
axes[0].set_title('Sagital')
axes[0].axis('off')

axes[1].imshow(data[:, y, :], cmap='gray', aspect=voxel_sizes[0]/voxel_sizes[2])
axes[1].set_title('Coronal')
axes[1].axis('off')

axes[2].imshow(data[:, :, z], cmap='gray') #, aspect=voxel_sizes[1]/voxel_sizes[0])
axes[2].set_title('Axial')
axes[2].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 6, figsize=[18, 3])

n = 0
slice = 0
for _ in range(6):
    ax[n].imshow(data[:, :, slice], 'gray')
    ax[n].set_xticks([])
    ax[n].set_yticks([])
    ax[n].set_title('Slice number: {}'.format(slice), color='r')
    n += 1
    slice += 10

fig.subplots_adjust(wspace=0, hspace=0)
plt.show()

##fMRI data

In [ ]:
# Basic information about the data acquisition
x_size = 64
y_size = 64
n_slice = 64
n_volumes = 96

In [ ]:
# Find all files in the data folder
data_path = './fMRI_data/fM00223/'
files = os.listdir(data_path)

In [ ]:
print(data.shape)

In [ ]:
files = sorted(os.listdir(data_path))

In [ ]:
files

La resolución en el plano de cada corte es mucho más baja que los 256x256 del estructural. La fMRI sacrifica resolución espacial para poder capturar volúmenes rápido y repetidamente. Ademas, se adquirieron en el tiempo 96 "fotos 3D" tomadas una tras otra.

In [ ]:
# Read in the data and organize it with respect to the acquisition parameters
data_all = []
for data_file in files:
    if data_file[-3:] == 'hdr':
        data = nibabel.load(data_path + data_file).get_fdata()
        data_all.append(data.reshape(x_size, y_size, n_slice))

In [ ]:
len(data_all)

In [ ]:
# Create a 3x6 subplot
fig, ax = plt.subplots(3, 6, figsize=[18, 11])

# Orgaize the data for visualisation in the coronal plane
coronal = np.transpose(data_all, [1, 3, 2, 0]) #reordenamiento de la lista
coronal = np.rot90(coronal, 1)

# Orgaize the data for visualisation in the transversal plane
transversal = np.transpose(data_all, [2, 1, 3, 0])
transversal = np.rot90(transversal, 2)

# Orgaize the data for visualisation in the sagittal plane
sagittal = np.transpose(data_all, [2, 3, 1, 0])
sagittal = np.rot90(sagittal, 1)

# Plot some of the images in different planes
n = 10
for i in range(6):
    ax[0][i].imshow(coronal[:, :, n, 0], cmap='gray')
    ax[0][i].set_xticks([])
    ax[0][i].set_yticks([])
    if i == 0:
        ax[0][i].set_ylabel('coronal', fontsize=25, color='r')
    n += 10

n = 5
for i in range(6):
    ax[1][i].imshow(transversal[:, :, n, 0], cmap='gray')
    ax[1][i].set_xticks([])
    ax[1][i].set_yticks([])
    if i == 0:
        ax[1][i].set_ylabel('transversal', fontsize=25, color='r')
    n += 10

n = 5
for i in range(6):
    ax[2][i].imshow(sagittal[:, :, n, 0], cmap='gray')
    ax[2][i].set_xticks([])
    ax[2][i].set_yticks([])
    if i == 0:
        ax[2][i].set_ylabel('sagittal', fontsize=25, color='r')
    n +=10
fig.subplots_adjust(wspace=0, hspace=0) #elimina espacios entre subplots
plt.show()

las 6 imágenes de cada fila NO muestran cómo cambia la actividad cerebral en el tiempo, muestran 6 cortes espaciales distintos (posiciones 10, 20, 30, 40, 50, 60 del eje correspondiente), pero siempre del mismo instante de tiempo (volumen 0, el primero de los 96).

##Señal fMRI
Volumen Temporal

In [ ]:
# Create an empty plot with defined aspect ratio
fig, ax = plt.subplots(1, 1, figsize=[18, 5])

# Plot the timecourse of a random voxel
ax.plot(transversal[30, 30, 35, :], lw=3)
ax.set_xlim([0, transversal.shape[3]-1])
ax.set_xlabel('time [s]', fontsize=20)
ax.set_ylabel('signal strength', fontsize=20)
ax.set_title('voxel time course', fontsize=25)
ax.tick_params(labelsize=12)

plt.show()

Un  voxel es el equivalente a un píxel de una imagen 2D pero en una imagen 3D. La señal fMRI es una grafica en la que se fijan las tres coordenadas espaciales y se tiene en cuenta, para ese punto del volumen, La intensidad del voxel con respecto al tiempo. Es decir, se toma un voxel para analizar como cambia su valor en el tiempo, "voxel time course".

Entonces,  cuando se nota un aumento en  la intensidad del voxel se puede decir que es un aumento del flujo sanguineo y por ende una excitacion en esa parte especifica del cerebro:

1. Cuando un grupo de neuronas se activa (sea excitación o inhibición), consume más energía.
2. El cerebro responde enviando más sangre oxigenada a esa zona — de hecho, envía más de la que la zona realmente necesita metabólicamente (hiperemia funcional).
3. Ese exceso de sangre oxigenada cambia las propiedades magnéticas locales del tejido, y eso es lo que el escáner de MRI detecta como un cambio de señal.

In [ ]:
x = int(input("Slice 0: "))  #0-255
y = int(input("Slice 1: "))  #0-255
z = int(input("Slice 2: "))  #0-5


fig, ax = plt.subplots(1, 1, figsize=[18, 5])

# Plot the timecourse of a random voxel
ax.plot(transversal[x, y, z, :], lw=3)
ax.set_xlim([0, transversal.shape[3]-1])
ax.set_xlabel('time [s]', fontsize=20)
ax.set_ylabel('signal strength', fontsize=20)
ax.set_title('voxel time course', fontsize=25)
ax.tick_params(labelsize=12)

plt.show()

Lo que mide la fMRI se llama señal BOLD (Blood Oxygen Level Dependent). Lo que detecta el escáner no es "cuánta sangre pasa", sino la proporción de hemoglobina oxigenada vs. desoxigenada en esa zona. La hemoglobina desoxigenada distorsiona el campo magnético local (es paramagnética) y reduce la señal; cuando llega sangre extra oxigenada, esa distorsión disminuye y la señal sube. Es una medida indirecta de actividad, mediada por hemodinámica, no una medida directa de disparos neuronales.



la señal BOLD tiene mucho ruido fisiológico (respiración, pulso cardíaco, movimiento de cabeza) que puede producir subidas y bajadas que no tienen nada que ver con actividad neuronal real.


*"Un aumento sostenido en la señal BOLD de un vóxel, desfasado unos segundos respecto al estímulo, y estadísticamente significativo al compararlo con un modelo esperado, sugiere un aumento en la demanda metabólica de esa zona — lo cual suele estar asociado con actividad neuronal (excitatoria o inhibitoria) en esa región"*







In [ ]:
#Guardar datos para no descomprimir zip de nuevo
data_all = np.transpose(data_all, [3, 2, 1, 0])
data_all = np.reshape(data_all, [n_slice, y_size*x_size, n_volumes])

# Check if output path exists, if not create it.
if os.path.exists('./fMRI_data/csv_data') == False:
    os.mkdir('./fMRI_data/csv_data')

# Export each slice as a .csv file
n = 0
for export in data_all:

    save_file = 'slice_{}.csv'.format(n)
    save_path = './fMRI_data/csv_data/{}'.format(save_file)
    pd.DataFrame(export).to_csv(save_path, header=False, index=False)
    n += 1